# 重現最終提交檔（最後一步）：競賽二 Rank（v13n）

AI CUP 2026 圍棋・競賽二 Rank（最終上傳 v13n）。本筆記本從保存的中間檔重現**最終提交檔的最後一步**，並印出 sha256 與預期值比對。

> **關於中間檔**：最後一步需要的「驗證題與測試題的六組機率」（`val_preds.npz`、`test_inputs.npz`）是由主辦方的官方資料算出來的（驗證側還含標籤），**不放在本 repo，也不放在 Release**。它們只提供給主辦方（私人連結），請先把檔案放到一個資料夾（例如掛載的 Google Drive），並把下面「設定」格的 `INTERMEDIATE_DIR` 指到該資料夾。沒有這些檔，請改看 `full_pipeline.ipynb`（從官方資料重新產生的步驟）。
> 我們自訓的權重（第二階段 `stage2_fit.pkl`、ft4 讀法網路、認人網路等）則在 GitHub Release，由 `RELEASE_URL` 下載。

| 步驟 | 內容 | 硬體 | 耗時 |
|---|---|---|---|
| 1–2 | clone repo、安裝鎖定版本套件 | CPU | 約 2–5 分鐘 |
| 3 | 外部預訓練權重（預設略過，最後一步用不到） | — | 下載約 1.1 GB（若開） |
| 4 | 從 GitHub Release 下載自訓權重（約 167 MB；最後一步只用其中 `stage2_fit.pkl` 1.8 MB）並驗 sha256 | CPU | 數十秒 |
| 5 | 從 `INTERMEDIATE_DIR` 讀入最後一步需要的中間檔（約 4.6 MB）並驗 sha256 | CPU | 數秒 |
| 7 | Rank 最後一步 | CPU | 約 1 分鐘 |
| 8 | sha256 比對 | CPU | 數秒 |

**Colab 免費版就能跑完，不需要 GPU。** `WEIGHTS = "final"` 只下載最後一步需要的那一個檔。

這份筆記本**只重現最後一步**。從棋譜整條重跑做不到公開重現的程度（上游中間產物已刪、重編約 70 小時，另外還要重訓認人網路與 ft4），分段說明與每段的 GPU 需求、耗時見 `colab/full_pipeline.ipynb` 與 `PIPELINE.md`。

In [ ]:
# 設定。標示「待補」的值要等 GitHub repo 與 Release 建好後填入。
import os, sys, subprocess
from pathlib import Path

REPO_URL    = os.environ.get("REPO_URL", "待補")        # 例：https://github.com/<帳號>/aicup2026-go-rank.git
RELEASE_URL = os.environ.get("RELEASE_URL", "待補")     # 我們自訓權重的下載網址，例：https://github.com/<帳號>/aicup2026-go-rank/releases/download/<tag>
INTERMEDIATE_DIR = os.environ.get("INTERMEDIATE_DIR", "/content/drive/MyDrive/aicup2026-go-rank-intermediate")   # 官方資料衍生的中間檔（只提供給主辦方，見上面說明）
REPO_DIR    = Path(os.environ.get("REPO_DIR", "/content/aicup2026-go-rank"))
# （環境變數只是方便本機測試；在 Colab 直接改上面的字串即可。）

def run(cmd, cwd=None, env=None, check=True):
    """執行指令並印出輸出（stdout＋stderr）。"""
    e = dict(os.environ, **(env or {}))
    r = subprocess.run(cmd, cwd=str(cwd or REPO_DIR), env=e, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(r.stdout[-6000:] if len(r.stdout) > 6000 else r.stdout)
    if check and r.returncode:
        raise RuntimeError(f"指令失敗（exit {r.returncode}）：{cmd}")
    return r


In [ ]:
# 1) 取得程式碼
if not (REPO_DIR / "colab").exists():
    assert REPO_URL != "待補", "請先把上一格的 REPO_URL 填成實際網址"
    run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], cwd="/")
print("repo:", REPO_DIR, "| commit:", run(["git", "rev-parse", "--short", "HEAD"], check=False).stdout.strip())


In [ ]:
# 2) 安裝鎖定版本的套件（requirements.txt；最後一步只需要 CPU）。
#    Colab 若提示「需重新啟動工作階段」就照做，然後從「設定」那格重跑。
run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
import importlib.metadata as md
need = {"numpy": "2.5.4", "scipy": "1.18.1", "pandas": "3.0.6", "scikit-learn": "1.9.1", "lightgbm": "4.7.0"}
bad = {k: (md.version(k), v) for k, v in need.items() if md.version(k) != v}
print("Python", sys.version.split()[0], "| 版本核對：", "全部符合" if not bad else f"不符 {bad}")
assert not bad, "scikit-learn 必須是 1.9.1（Player 的候選池分類頭只在此版本位元重現），其他套件請照 requirements.txt"


In [ ]:
# 3) 外部預訓練權重（KataGo 執行檔／網路、HumanSL .ckpt、Strength Estimator）。
#    本 repo 與 Release 都不散布它們；這一格只從官方網址下載（並驗 sha256）。「最後一步重現」用不到，預設不下載（合計約 1.1 GB）。
#    要重新抽特徵或重新訓練（full_pipeline.ipynb）才需要。來源網址、版本、授權見 README「外部資源」。
DOWNLOAD_EXTERNAL = False
sys.path.insert(0, str(REPO_DIR / "colab"))
import colab_utils as U
if DOWNLOAD_EXTERNAL:
    U.fetch_external(REPO_DIR)
else:
    for k, e in U.EXTERNAL.items():
        print(f"  （略過）{k}: {e['url']}  [{e['bytes']:,} bytes, sha256 {e['sha256'][:12]}…, {e['license']}]")


In [ ]:
# 4) 我們自訓的權重：從 GitHub Release 下載並驗 sha256（清單見 ASSETS.md、colab/assets_manifest.json 的 category == "weights"）。
#    WEIGHTS = "all"：全部自訓權重；"final"：只下載最後一步需要的；"none"：不下載。
WEIGHTS = "all"
if WEIGHTS != "none":
    U.fetch_weights(REPO_DIR, RELEASE_URL, final_only=(WEIGHTS == "final"))


In [ ]:
# 5) 官方資料衍生的中間檔：從 INTERMEDIATE_DIR 讀取（只提供給主辦方；本 repo 與 Release 都沒有）。
#    在 Colab 可把主辦方給的私人連結內容放到自己的 Google Drive，掛載後把 INTERMEDIATE_DIR 指到該資料夾。
if not Path(INTERMEDIATE_DIR).is_dir():
    try:
        from google.colab import drive; drive.mount("/content/drive")      # 只在 Colab 有效
    except Exception as e:
        print("（不在 Colab，或掛載失敗）", e)
U.fetch_intermediate(REPO_DIR, INTERMEDIATE_DIR, final_only=True)      # 只取最後一步需要的；每個檔都驗 sha256


In [ ]:
# 6) 預期的 sha256（原專案上傳檔；2026-11-26 實測）
EXPECT = {
    "rank_v13n":  "3a9525a3e7e6273727d0e5fd0a6fc7d5fc2921b17c7687b36f83e961c9d13919",   # Rank 最終上傳檔
}
OUT_DIR = REPO_DIR / "out"; OUT_DIR.mkdir(exist_ok=True)
ENV = {"PYTHONPATH": "src:scripts:.", "WORK_DIR": str(REPO_DIR / ".scratch")}
def fresh(name):                                   # 腳本不覆寫既有輸出，重跑時先刪
    p = OUT_DIR / name
    if p.exists(): p.unlink()
    return str(p)


In [ ]:
# 7) Rank 最後一步（約 1 分鐘，CPU）：六組機率 → 第二階段 → v13n
#    輸入：.scratch/e103/val_preds.npz、.scratch/e117/final/test_inputs.npz（中間檔）、.scratch/e117/final/stage2_fit.pkl（自訓權重）
#    輸出：out/rank_v13n.csv（800 列 question_id,pred_rank）
run([sys.executable, "final/make_rank_final.py", "--out", fresh("rank_v13n.csv")], env=ENV)

In [ ]:
# 比對 sha256
import hashlib
sha = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()
rows = [("Rank v13n", OUT_DIR / "rank_v13n.csv", EXPECT["rank_v13n"])]
ok = True
for name, p, want in rows:
    got = sha(p); same = got == want; ok &= same
    print(f"{'OK ' if same else 'DIFF'} {name:22s} {got}\n{'':27s}預期 {want}")
print("\n全部位元組相同" if ok else "\n有檔案不同：請看上面各步驟的輸出（版本、sha256 下載驗證、官方資料）")
assert ok
